# YOLOv11 Experiment Tracking — Surgical Instrument Detection

Turns one training run into a self-contained, browsable experiment record: real prediction images,
real ground-truth images, real detection metrics — not just a `best.pt` with no context.

**Iron rule (ห้ามแก้ไข/ลบไฟล์ใน Dataset ต้นฉบับเด็ดขาด)**: `DATASET_DIR` below is READ-ONLY. Every
mutation in this notebook goes through `train_roboflow_yolo.py`'s already-verified non-destructive
balancing (`copy_train_split_for_balancing` + `balance_train_split(protected_dir=DATASET_DIR)`) —
the same mechanism proven this session with a real byte-for-byte hash comparison, not a new promise.

**Process**: this notebook was first verified end-to-end with a short debug pass (2 epochs, tiny
image size) — confirmed every one of the 12 steps below works correctly, byte-for-byte confirmed
`DATASET_DIR` stays untouched. The config below now reflects a real training attempt, run only
after the user's explicit go-ahead.

In [ ]:
import sys, time, json, shutil, datetime
from pathlib import Path

import cv2
from IPython.display import Image, display

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from train_roboflow_yolo import (
    load_data_yaml, resolve_split_images_dir, labels_dir_for, parse_labels_in_dir,
    compute_class_counts, copy_train_split_for_balancing, balance_train_split, write_working_data_yaml,
)
from detector import CLASS_NAMES
import experiment_tracking as et

# The one hard rule this whole notebook exists to respect: DATASET_DIR is READ-ONLY.
# copy_train_split_for_balancing()/balance_train_split(protected_dir=...) below already guarantee
# this at runtime (a real assertion, not just a comment) -- verified this session with a real
# byte-for-byte hash comparison. Never call .unlink()/.write_*()/cv2.imwrite() on anything under
# DATASET_DIR directly in this notebook.
DATASET_DIR = Path(r"C:\Users\USER\OneDrive\Documents\BEAM\VideoFrameExtractor\ssid.v6i.yolov11")
EXPERIMENTS_ROOT = (NOTEBOOK_DIR / "runs" / "model_experiments").resolve()
TMP_RUN_DIR = EXPERIMENTS_ROOT / "_tmp_run"
EXPERIMENTS_ROOT.mkdir(parents=True, exist_ok=True)

# -- Config. Debug pass (epochs=2, imgsz=320) already verified end-to-end on 2026-08-08; these are
# now real-run values, run with the user's explicit go-ahead. Adjust here between attempts.
MODEL = "yolo11n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 8
DEVICE = "cpu"
SEED = 42
CONF = 0.25
IMBALANCE_THRESHOLD = 1.5
MAX_DROP_FRACTION = 0.9

print(f"DATASET_DIR (read-only): {DATASET_DIR}")
print(f"EXPERIMENTS_ROOT: {EXPERIMENTS_ROOT}")
print(f"Debug config: model={MODEL} epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} device={DEVICE}")

## Step 1 — Load dataset (read-only)

In [ ]:
yaml_data = load_data_yaml(DATASET_DIR)
class_names = yaml_data["names"]
assert class_names == CLASS_NAMES, f"Dataset class order {class_names} != detector.py CLASS_NAMES {CLASS_NAMES}"

orig_train_images = resolve_split_images_dir(DATASET_DIR, yaml_data, "train")
orig_train_labels = labels_dir_for(orig_train_images)
orig_val_images = resolve_split_images_dir(DATASET_DIR, yaml_data, "val")
orig_val_labels = labels_dir_for(orig_val_images) if orig_val_images else None
orig_test_images = resolve_split_images_dir(DATASET_DIR, yaml_data, "test")

print(f"classes: {class_names}")
print(f"train images: {orig_train_images} ({len(list(orig_train_images.iterdir()))} files)")
print(f"val images:   {orig_val_images} ({len(list(orig_val_images.iterdir())) if orig_val_images else 0} files)")
print(f"test images:  {orig_test_images} ({len(list(orig_test_images.iterdir())) if orig_test_images else 0} files)")

## Step 2 — Class counts

In [ ]:
train_records = parse_labels_in_dir(orig_train_images, orig_train_labels, "train", class_names)
train_counts = compute_class_counts(train_records)

print("Train class counts (from the ORIGINAL dataset, read-only):")
for name in class_names:
    print(f"  {name:15s} {train_counts.get(name, 0)}")

## Step 3 — Annotation examples (ground truth, before training)

In [ ]:
samples = et.draw_sample_annotations(orig_train_images, orig_train_labels, class_names, n=6)
print(f"{len(samples)} annotation examples (ground truth, drawn with detector.py's own draw_boxes):")
for s in samples:
    ok, buf = cv2.imencode(".jpg", s["image"])
    display(Image(data=buf.tobytes()))
    print(s["name"])

## Step 4 — Train

Balances a working COPY of the train split only (never `DATASET_DIR` itself — `protected_dir=`
below is a real runtime assertion), then trains with Ultralytics.

In [ ]:
if TMP_RUN_DIR.exists():
    shutil.rmtree(TMP_RUN_DIR)

# Non-destructive: balancing only ever touches this working copy, never DATASET_DIR itself --
# protected_dir=DATASET_DIR below is a hard runtime assertion, not just a comment.
train_copy_images, train_copy_labels = copy_train_split_for_balancing(orig_train_images, orig_train_labels, TMP_RUN_DIR)
copy_records = parse_labels_in_dir(train_copy_images, train_copy_labels, "train", class_names)
balance_report = balance_train_split(copy_records, seed=SEED, max_drop_fraction=MAX_DROP_FRACTION,
                                      imbalance_threshold=IMBALANCE_THRESHOLD, protected_dir=DATASET_DIR)
print(f"[balance] {balance_report}")

work_yaml = write_working_data_yaml(yaml_data, train_copy_images, orig_val_images, orig_test_images, TMP_RUN_DIR)

from ultralytics import YOLO
model = YOLO(MODEL)
train_start = time.time()
model.train(data=str(work_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, seed=SEED,
            project=str(TMP_RUN_DIR), name="train", exist_ok=True)
training_seconds = time.time() - train_start

best_weights = TMP_RUN_DIR / "train" / "weights" / "best.pt"
assert best_weights.exists(), f"Training finished but best.pt not found at {best_weights}"
print(f"[train] done in {training_seconds:.1f}s — best weights at {best_weights}")

## Step 5 — Training curves

In [ ]:
results_png = TMP_RUN_DIR / "train" / "results.png"
if results_png.exists():
    display(Image(filename=str(results_png)))
else:
    print("results.png not found (unexpected for a completed run)")

## Step 6 — Confusion matrix

In [ ]:
cm_png = TMP_RUN_DIR / "train" / "confusion_matrix.png"
if cm_png.exists():
    display(Image(filename=str(cm_png)))
else:
    print("confusion_matrix.png not found -- can happen with a very small/edge-case val set")

## Step 7 — Detection metrics

Real Ultralytics `DetMetrics` (precision/recall/mAP50/mAP50-95, overall + per-class) — **not**
sklearn `classification_report`, since this is object detection, not single-label classification.

In [ ]:
val_model = YOLO(str(best_weights))
val_metrics = val_model.val(data=str(work_yaml), device=DEVICE, verbose=False)

print(f"Precision:  {val_metrics.box.mp:.4f}")
print(f"Recall:     {val_metrics.box.mr:.4f}")
print(f"mAP50:      {val_metrics.box.map50:.4f}")
print(f"mAP50-95:   {val_metrics.box.map:.4f}")
print()
print("Per-class:")
for i, cid in enumerate(getattr(val_metrics.box, "ap_class_index", [])):
    name = class_names[int(cid)] if int(cid) < len(class_names) else str(cid)
    ap50 = val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else float("nan")
    ap = val_metrics.box.ap[i] if i < len(val_metrics.box.ap) else float("nan")
    print(f"  {name:15s} mAP50={ap50:.4f}  mAP50-95={ap:.4f}")

## Step 8 — Real prediction images (validation set)

In [ ]:
n_pred, n_gt = et.render_val_visualizations(best_weights, orig_val_images, orig_val_labels, class_names,
                                             TMP_RUN_DIR, conf=CONF, device=DEVICE)
print(f"Wrote {n_pred} prediction images + {n_gt} ground-truth images.")

pred_dir = TMP_RUN_DIR / "val_predictions"
print("\nSample predictions:")
for p in sorted(pred_dir.iterdir())[:4]:
    display(Image(filename=str(p)))
    print(p.name)

## Step 9 — Real ground-truth images (same validation images, for direct comparison)

In [ ]:
gt_dir = TMP_RUN_DIR / "val_ground_truth"
print("Sample ground truth (same images as Step 8, for direct comparison):")
for p in sorted(gt_dir.iterdir())[:4]:
    display(Image(filename=str(p)))
    print(p.name)

## Step 10 — Training time

In [ ]:
print(f"Training time: {training_seconds:.1f} seconds ({training_seconds/60:.1f} minutes)")

## Step 11 — Summary of all runs so far

In [ ]:
history_rows = et.read_experiment_history(EXPERIMENTS_ROOT)
if not history_rows:
    print("No runs recorded yet in experiment_history.csv (this will be the first one, after Step 12 below).")
else:
    header = ["run_id", "mAP50", "mAP50_95", "precision", "recall", "training_time", "model_path"]
    print(" | ".join(f"{h:>12s}" for h in header))
    for row in history_rows:
        print(" | ".join(f"{row.get(h, ''):>12s}" for h in header))

## Step 12 — Auto-save to `<mAP50>_percent/`

Renames the temp run dir to its final score-named folder, copies Ultralytics' own artifacts up
flat (matching the requested layout, not nested under a `train/` subfolder), writes
`metrics.json`/`config.json`/`training_time.json`, and appends one row to `experiment_history.csv`.

In [ ]:
map50 = float(val_metrics.box.map50)

final_dir = et.finalize_run_dir(TMP_RUN_DIR, EXPERIMENTS_ROOT, map50)
copied = et.collect_run_artifacts(final_dir / "train", final_dir)
print(f"Copied artifacts: {copied}")

# The nested train/ subfolder's contents are now duplicated at the flat level above -- remove it,
# along with the balanced working copy and the temp data.yaml (both were only ever needed to drive
# training itself), so the final layout matches the requested one exactly, no leftover clutter.
shutil.rmtree(final_dir / "train", ignore_errors=True)
shutil.rmtree(final_dir / "train_balanced", ignore_errors=True)
(final_dir / "data.yaml").unlink(missing_ok=True)

metrics_data = et.write_metrics_json(val_metrics, final_dir, class_names)
config_data = {
    "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "device": DEVICE, "seed": SEED,
    "conf": CONF, "imbalance_threshold": IMBALANCE_THRESHOLD, "max_drop_fraction": MAX_DROP_FRACTION,
    "balancing_method": et.BALANCING_METHOD, "balance_report": balance_report,
    "dataset_dir": str(DATASET_DIR),
}
et.write_config_json(config_data, final_dir)
time_data = et.write_training_time_json(training_seconds, final_dir)

run_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
history_path = et.append_experiment_history(EXPERIMENTS_ROOT, {
    "run_id": run_id, "model": MODEL, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "device": DEVICE,
    "augmentation": False, "balancing_method": et.BALANCING_METHOD,
    "precision": round(metrics_data["precision"], 4), "recall": round(metrics_data["recall"], 4),
    "mAP50": round(metrics_data["mAP50"], 4), "mAP50_95": round(metrics_data["mAP50_95"], 4),
    "training_time": time_data["human_readable"], "model_path": str(final_dir / "best.pt"),
})

print(f"\n✅ Run saved to: {final_dir}")
print(f"✅ experiment_history.csv updated: {history_path}")
print("\nFiles in this run's folder:")
for p in sorted(final_dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(final_dir)}")

## Model selection / pruning (optional — run this cell yourself, deliberately)

This does **not** run automatically after training. It scans every run under `EXPERIMENTS_ROOT`,
and deletes **only** `best.pt`/`last.pt` (nothing else — metrics/images/csv/config all stay) for
any run below the mAP50 threshold that isn't the current single best run. Every deletion is
asserted to stay inside `EXPERIMENTS_ROOT` — it can never touch anything else, including
`DATASET_DIR`.

Set `RUN_PRUNING = True` below and run the cell only when you actually want to prune.

In [ ]:
RUN_PRUNING = False  # flip to True and re-run this cell when you actually want to prune

if RUN_PRUNING:
    result = et.prune_low_scoring_runs(EXPERIMENTS_ROOT, threshold=0.80)
    print(f"Best run so far: {result['best_run']} (mAP50={result['best_score']})")
    print(f"Kept weights for {len(result['kept'])} run(s).")
    print(f"Pruned {len(result['pruned'])} weight file(s):")
    for p in result["pruned"]:
        print(f"  {p}")
else:
    print("RUN_PRUNING is False -- nothing changed. Set it to True above to actually prune.")